In [10]:
import qr
import importlib
importlib.reload(qr)

<module 'qr' from '/home/lexu/University/visible-light/inbedded/qr.py'>

In [11]:
import math
import numpy as np
import segno
import cv2
import moderngl

In [12]:
def create_qr(content):
    qr = segno.make_qr(content, version=10, mode="byte")
    matrix = qr.matrix
    arr = np.array(matrix, dtype=int)

    qr.save("output/qr.png", scale=10)

In [13]:
qcd = cv2.QRCodeDetector()

import zxingcpp

def read_qr(img):
    results = zxingcpp.read_barcodes(
        img,
        formats=zxingcpp.BarcodeFormat.QRCode
    )
    return [result.bytes for result in results]

In [ ]:
import cv2
import segno
import numpy as np
from pathlib import Path
from protocol import *

BACKGROUND_PATH = "minecraft.jpg"
OUTPUT_PATH = "output/imperceptible_qr.mp4"

VIDEO_WIDTH = 1920
VIDEO_HEIGHT = 1080
FPS = 60
DURATION = 20
PHASE_DURATION = 1/15

STRENGTH = 0.02  # Brightness change: +/- 10.2 / 255
QR_MODULE_SIZE = 10

message = SendSubtitle(
    currentTime=1000,
    subtitleStartTime=1200,
    subtitleEndTime=3500,
    sub_string="Hello World!"
)

# Encode the protocol message
payload = encode_message(message)

# Create QR code with quiet zone
qr = segno.make_qr(payload, error="H", boost_error=False)
qr_bits = np.array(qr.matrix, dtype=np.uint8)
qr_bits = np.pad(qr_bits, 4, constant_values=0)

# Convert modules to a full-resolution QR
qr_bits = cv2.resize(
    qr_bits,
    None,
    fx=QR_MODULE_SIZE,
    fy=QR_MODULE_SIZE,
    interpolation=cv2.INTER_NEAREST
)

qr_h, qr_w = qr_bits.shape

# Load the static background
bg = cv2.imread(BACKGROUND_PATH)
if bg is None:
    raise FileNotFoundError(BACKGROUND_PATH)

bg = cv2.resize(bg, (VIDEO_WIDTH, VIDEO_HEIGHT))

if qr_h > VIDEO_HEIGHT or qr_w > VIDEO_WIDTH:
    raise ValueError("QR code is larger than the video")

# Center QR in background
x = (VIDEO_WIDTH - qr_w) // 2
y = (VIDEO_HEIGHT - qr_h) // 2

# +1 for black modules; -1 for white modules
qr_signal = np.where(qr_bits == 1, 1.0, -1.0)

def make_phase(phase):
    image = bg.copy()
    region = image[y:y+qr_h, x:x+qr_w].astype(np.float32)

    region += phase * STRENGTH * 255 * qr_signal[:, :, None]

    image[y:y+qr_h, x:x+qr_w] = np.clip(
        region, 0, 255
    ).astype(np.uint8)

    return image

def make_phase_hsl(phase):
    image = bg.copy()
    region = image[y:y+qr_h, x:x+qr_w]
    hsv_region = cv2.cvtColor(region, cv2.COLOR_BGR2HLS)
    h, v, s = cv2.split(hsv_region)

    v_float = v.astype(np.float32) + phase * STRENGTH * 255 * qr_signal
    v_modified = np.clip(v_float, 0, 255).astype(np.uint8)

    hsv_region = cv2.merge([h, v_modified, s])
    region = cv2.cvtColor(hsv_region, cv2.COLOR_HLS2BGR)

    image[y:y+qr_h, x:x+qr_w] = region.astype(np.uint8)

    return image

# Precompute the two phases
phase_a = make_phase_hsl(+1)
phase_b = make_phase_hsl(-1)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (VIDEO_WIDTH, VIDEO_HEIGHT)
)

if not writer.isOpened():
    raise RuntimeError("Could not create video")

for i in range(int(FPS * DURATION)):
    t = i / FPS
    phase = int(t / PHASE_DURATION) % 2
    writer.write(phase_a if phase == 0 else phase_b)

writer.release()

print(f"Saved: {OUTPUT_PATH}")
print(f"QR size: {qr_w}x{qr_h}")
print(f"QR position: ({x}, {y})")

# Preview original background and one modulated phase
cv2.imwrite("output/phase_a.png", phase_a)
cv2.imwrite("output/phase_b.png", phase_b)


Saved: output/imperceptible_qr.mp4
QR size: 410x410
QR position: (755, 335)


True

In [ ]:
import cv2
import numpy as np
import zxingcpp

# This should be the Irudin camera index
CAMERA_INDEX = 4

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
cap.set(cv2.CAP_PROP_FPS, 30)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

decoder = MessageDecoder()
previous = None

print("Camera FPS:", cap.get(cv2.CAP_PROP_FPS))

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (960, 540))

        if previous is not None:
            # Signed difference preserves bright/dark polarity
            diff = gray.astype(np.float32) - previous.astype(np.float32)

            # Threshold at zero to recover QR modules
            qr = np.where(diff > 0, 255, 0).astype(np.uint8)

            # Try both phase orders
            for candidate in (qr, cv2.bitwise_not(qr)):
                # Using this library because it returns bytes correctly unlike OpenCV
                results = zxingcpp.read_barcodes(
                    candidate,
                    formats=zxingcpp.BarcodeFormat.QRCode
                )

                for result in results:
                    for message in decoder.push(result.bytes):
                        print("Decoded:", message)

            cv2.imshow("Reconstructed QR", qr)

        previous = gray.copy()

        cv2.imshow("Camera", cv2.resize(frame, (960, 540)))

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


Camera FPS: 30.0
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime